In [1]:
# Setup: add project root
import sys
import os
sys.path.insert(0, '..')

from src.orchestrator import Orchestrator
from src.standards import METADATA_STANDARDS
from src.context.context_factory import create_context
BASE = os.path.abspath(os.path.join('..'))

import logging

# Setup logging for Jupyter notebook
# Force reconfigure by removing existing handlers
logger = logging.getLogger()
logger.setLevel(logging.INFO)

# Remove existing handlers to avoid duplicates
for handler in logger.handlers[:]:
    logger.removeHandler(handler)

# Add a fresh StreamHandler that outputs to stdout (visible in notebook)
handler = logging.StreamHandler(sys.stdout)
handler.setLevel(logging.INFO)
handler.setFormatter(logging.Formatter('%(asctime)s - %(levelname)s - %(name)s - %(message)s'))
logger.addHandler(handler)

print("Imports OK")

Imports OK


In [2]:
from src.orchestrator import run_metadata_extraction
multi_source = {
    'biota': os.path.join(BASE, 'data/biota_dataset/biota.csv'),
    'samples': os.path.join(BASE, 'data/biota_dataset/samples.csv'),
    'species': os.path.join(BASE, 'data/biota_dataset/species.csv'),
}

In [3]:
data_context = create_context(
    source=multi_source,
    name='biota_multi'
)
metadata_standard=METADATA_STANDARDS['spatial_ecological']
orchestrator = Orchestrator(topology_name='fast')
plan = orchestrator.generate_plan(
    context=data_context,
    metadata_standard=metadata_standard
)

2026-04-14 11:40:49,572 - INFO - root - PlanExecutor initialized with topology: fast
2026-04-14 11:40:49,573 - INFO - root -   Players per step: 2
2026-04-14 11:40:49,573 - INFO - root -   Debate rounds: 1
2026-04-14 11:40:49,574 - INFO - root -   Player pool: ['data_analyst', 'schema_expert']
2026-04-14 11:40:49,574 - INFO - root - Orchestrator initialized with topology: fast
2026-04-14 11:40:49,574 - INFO - root - ============================================================
2026-04-14 11:40:49,574 - INFO - root - GENERATING PLAN
2026-04-14 11:40:49,575 - INFO - root - Context: biota_multi
2026-04-14 11:40:49,575 - INFO - root - Context type: csv
2026-04-14 11:40:49,575 - INFO - root - Resources: ['biota', 'samples', 'species']
2026-04-14 11:40:49,575 - INFO - root - Multi-resource: True
2026-04-14 11:40:49,575 - INFO - root - ============================================================
2026-04-14 11:40:49,575 - INFO - root - Auto-added 'metadata_generator' for final metadata generati

In [4]:
# Validate plan dataflow
from src.orchestrator.utils import validate_plan_dataflow

if plan:
    # Convert to dict list for validation
    plan_dicts = plan.to_dict_list()
    
    is_valid, message = validate_plan_dataflow(plan_dicts)
    
    if is_valid:
        print(f"✅ {message}")
    else:
        print(f"❌ {message}")
else:
    print("No plan to validate.")

✅ Plan dataflow is valid.


In [5]:
plan.pretty_print()

Plan Steps:
Step 0: get_field_statistics
  Rationale: Analyze all three resources to understand field distributions, data quality, and characteristics needed for metadata generation
  Required Artifacts: {}
  Produced Artifacts: ['biota:field_stats', 'samples:field_stats', 'species:field_stats']
Step 1: get_relationships
  Rationale: Discover and validate relationships between resources to understand data structure for metadata
  Required Artifacts: {}
  Produced Artifacts: ['relationships']
Step 2: generate_metadata
  Rationale: Generate complete metadata using all gathered analysis and relationship information
  Required Artifacts: {'metadata_standard': 'metadata_standard', 'biota_stats': 'biota:field_stats', 'samples_stats': 'samples:field_stats', 'species_stats': 'species:field_stats', 'relationships': 'relationships'}
  Produced Artifacts: ['metadata_output']


In [6]:
from src.orchestrator.plan_executor import PlanExecutor
from src.tools.context_tools import register_context

context_key = "ctx_biota_multi"
register_context(context_key, data_context)
metadata_standard = METADATA_STANDARDS["spatial_ecological"]
executor = PlanExecutor(topology_name="fast")
result = executor.execute(
    plan=plan,
    context=data_context,
    context_key=context_key,
    metadata_standard=metadata_standard,
    metadata_standard_name="spatial_ecological"
)

2026-04-14 11:43:48,282 - INFO - root - PlanExecutor initialized with topology: fast
2026-04-14 11:43:48,282 - INFO - root -   Players per step: 2
2026-04-14 11:43:48,283 - INFO - root -   Debate rounds: 1
2026-04-14 11:43:48,284 - INFO - root -   Player pool: ['data_analyst', 'schema_expert']
2026-04-14 11:43:48,285 - INFO - root - Using structured output schema: SpatialEcologicalMetadata
2026-04-14 11:43:48,285 - INFO - root - ============================================================
2026-04-14 11:43:48,285 - INFO - root - STARTING PLAN EXECUTION
2026-04-14 11:43:48,286 - INFO - root - Context: biota_multi
2026-04-14 11:43:48,286 - INFO - root - Type: csv
2026-04-14 11:43:48,286 - INFO - root - Resources: ['biota', 'samples', 'species']
2026-04-14 11:43:48,286 - INFO - root - Steps: 3
2026-04-14 11:43:48,287 - INFO - root - ============================================================
2026-04-14 11:43:48,287 - INFO - root - 
2026-04-14 11:43:48,288 - INFO - root - =================

In [7]:
from pprint import pprint
pprint(result.final_workspace['metadata_output'])

{'description': 'Multi-resource dataset containing benthic macrofauna '
                'abundance and biomass measurements from the Wadden Sea '
                'region, Netherlands. Includes 391,018 biota observations '
                'linked to 51,851 sampling events and 177 species taxonomic '
                'records.',
 'format': 'CSV (Comma-Separated Values) - 3 files: biota.csv (10.6 MB), '
           'samples.csv (4.4 MB), species.csv (17.9 KB)',
 'methods': 'Standardized benthic sampling using grid (85%) and random (15%) '
            'sampling types. Primary platform: boat (93%). Measurements '
            'include abundance per m² and ash-free dry mass (AFDM) per m². '
            'Taxonomic identification to species level (62.7%) or higher '
            'taxonomic groups.',
 'spatial_coverage': 'Wadden Sea region, Netherlands (52.90°N-53.53°N, '
                     '4.80°E-7.23°E). Includes tidal basins: Marsdiep, '
                     'Eierlandse Gat, Vlie, and associa